# Cyclospora outbreak

## CDC Surveillance data

In [1]:
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pandas as pd
import re

In [2]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)

In [3]:
page = await browser.new_page()

In [4]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [5]:
async with page.expect_download() as download_info:
    await page.click('a[aria-label="Download this data in a CSV file format."]')

download = await download_info.value
await download.save_as("data/data.csv")
print("Downloaded to:", await download.path())

Downloaded to: /var/folders/7_/ltwct_mx5q5c5xzx5gnl5p3m0000gr/T/playwright-artifacts-kIaX4O/ff807fb3-bcfc-428b-aad1-c18b1fd5d1db


In [6]:
df = pd.read_csv("data/data.csv")

In [7]:
df.head()

,Location,Number of Sick People
0,Alaska,1 to 10
1,Arkansas,1 to 10
2,California,1 to 10
3,Colorado,1 to 10
4,Connecticut,1 to 10


In [8]:
df['min'] = df['Number of Sick People'].str.extract(r'(\d+)\s+to\s+\d+').astype(int)
df['max'] = df['Number of Sick People'].str.extract(r'\d+\s+to\s+(\d+)').astype(int)

In [9]:
df['average'] = df[['min', 'max']].mean(axis=1)

In [10]:
df.sort_values(by='average', ascending=False, inplace=True)
df.head()

,Location,Number of Sick People,min,max,average
15,Michigan,161 to 300,161,300,230.5
20,New York,81 to 160,81,160,120.5
7,Illinois,31 to 80,31,80,55.5
21,North Carolina,31 to 80,31,80,55.5
11,Kentucky,31 to 80,31,80,55.5


In [11]:
df.to_csv("data/clean_data.csv", index=False)

## Michigan data

In [12]:
await page.goto("https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks")

<Response url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' request=<Request url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' method='GET'>>

In [13]:
total_cases = await page.inner_text('p:has(strong:text("Total Cases:"))')
print(total_cases)


Total Cases: 3,309
As of July 9, 2026, 44 reported cases indicated they had been hospitalized.


In [18]:
match = re.search(r'Total Cases:\s*([\d,]+)', total_cases)
cases = match.group(1)  # "3,309"

# optional: convert to integer
michigan_cases = int(cases.replace(',', ''))  # 3309

print(cases)      # 3,309
print(michigan_cases)  # 3309

3,309
3309


In [15]:
last_updated = await page.inner_text('p:has(strong:text("Last updated:"))')
print(last_updated)

Last updated: July 14, 2026


## Combine data

In [22]:
df

,Location,Number of Sick People,min,max,average
15,Michigan,161 to 300,161,300,230.5
20,New York,81 to 160,81,160,120.5
7,Illinois,31 to 80,31,80,55.5
21,North Carolina,31 to 80,31,80,55.5
11,Kentucky,31 to 80,31,80,55.5
26,Texas,31 to 80,31,80,55.5
19,New Jersey,31 to 80,31,80,55.5
6,Georgia,11 to 30,11,30,20.5
5,Florida,11 to 30,11,30,20.5
8,Indiana,11 to 30,11,30,20.5


In [23]:
new_row = {
    'Location': 'Michigan',
    'Number of Sick People': michigan_cases,
    'min': michigan_cases,
    'max': michigan_cases,
    'average': michigan_cases
}

df_local = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [26]:
df_local

,Location,Number of Sick People,min,max,average
0,Michigan,161 to 300,161,300,230.5
1,New York,81 to 160,81,160,120.5
2,Illinois,31 to 80,31,80,55.5
3,North Carolina,31 to 80,31,80,55.5
4,Kentucky,31 to 80,31,80,55.5
5,Texas,31 to 80,31,80,55.5
6,New Jersey,31 to 80,31,80,55.5
7,Georgia,11 to 30,11,30,20.5
8,Florida,11 to 30,11,30,20.5
9,Indiana,11 to 30,11,30,20.5


In [21]:
df_local.to_csv("data/clean_data_local.csv", index=False)